# 🇳🇬 The African Election AI Shootout: Auditing Nigerian Form EC8A Result Sheets with Qwen2.5-VL
### Large-Scale Forensic Audit on Authentic INEC IReV Polling Unit Result Sheets

This notebook runs an automated, forensic evaluation of **Qwen2.5-VL** (the world's leading open-weight multimodal vision model) across authentic photographed Form EC8A result sheets from the Nigerian Independent National Electoral Commission (INEC) IReV portal.

---
### ⚡ Quick Start:
1. Make sure your Colab runtime is set to **T4 GPU** (`Runtime` -> `Change runtime type` -> select `T4 GPU`).
2. Click **Runtime** -> **Run all** (or press `Ctrl+F9`).
3. Sit back and watch the model audit hundreds of polling units and generate the final accuracy scorecard!

## Step 1: Install Dependencies & Setup Environment

In [ ]:
!pip install -q transformers accelerate torchvision qwen-vl-utils bitsandbytes requests tqdm pandas matplotlib seaborn

import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from qwen_vl_utils import process_vision_info

print("Loading Qwen2.5-VL-7B-Instruct onto GPU...")
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4"
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    quantization_config=bnb_config
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(" Model loaded successfully on GPU:", torch.cuda.get_device_name(0))


In [ ]:
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print("Loading Qwen2.5-VL-7B-Instruct onto GPU...")
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
    device_map="auto",
    load_in_4bit=True
)
processor = AutoProcessor.from_pretrained(MODEL_ID)
print(" Model loaded successfully on GPU:", torch.cuda.get_device_name(0))

## Step 3: Fetch Real Form EC8A Data & Human Ground Truth
*We stream the verified election dataset from the Osun State Governorship election archive.*

In [ ]:
import requests
import json

INDEX_URL = "https://raw.githubusercontent.com/IamDejman/osun-decides/main/data/results.json"
print("Fetching election database from GitHub...")
resp = requests.get(INDEX_URL)
data = resp.json()

# Extract all polling units with real camera uploads and human ground truth
all_pus = []
for lga in data['lgas']:
    for ward in lga['wards']:
        for pu in ward['pus']:
            if pu.get('status') == 'transcribed' and pu.get('img') and pu.get('votes'):
                if len(pu['votes']) >= 5:
                    all_pus.append({
                        "puCode": pu['code'],
                        "puName": pu['name'],
                        "ward": ward['name'],
                        "lga": lga['name'],
                        "imageUrl": pu['img'],
                        "groundTruth": pu['votes']
                    })

print(f" Found {len(all_pus)} fully transcribed authentic polling units!")

# SET SAMPLE SIZE (e.g. 100, 250, 500, or 1000)
NUM_SHEETS = 250   # Adjust as desired
test_batch = all_pus[:NUM_SHEETS]
print(f"Selected benchmark batch size: {len(test_batch)} sheets across Osun State LGAs.")

## Step 4: Run the Forensic Vision Audit Engine
*Processes each authentic photographed result sheet, performs blind digit extraction, and scores exact matches, hallucinations, and legal math compliance.*

In [ ]:
import os
import re
import time
from PIL import Image
from io import BytesIO
from tqdm import tqdm
import pandas as pd

PARTIES = ["A", "AA", "AAC", "ADC", "ADP", "APC", "APGA", "APM", "APP", "BP", "NNPP", "PRP", "SDP", "YPP", "ZLP"]

SYSTEM_PROMPT = """You are a forensic document auditor reading an authentic Nigerian Form EC8A (Polling Unit Election Result Sheet). 
Carefully inspect the handwritten numbers in the table for each political party:
A, AA, AAC, ADC, ADP, APC, APGA, APM, APP, BP, NNPP, PRP, SDP, YPP, ZLP.

Rules:
1. For each party, extract the integer number of votes written in figures. If the box has 0, NIL, or a dash, write 0. If unreadable, write null.
2. Also extract accredited_voters, valid_votes, rejected_votes, and total_votes_cast.

Respond ONLY with a valid JSON object matching this schema:
{
  "votes": {
    "A": 0, "AA": 0, "AAC": 0, "ADC": 0, "ADP": 0, "APC": 0, "APGA": 0, "APM": 0, 
    "APP": 0, "BP": 0, "NNPP": 0, "PRP": 0, "SDP": 0, "YPP": 0, "ZLP": 0
  },
  "accredited": 0,
  "valid": 0,
  "rejected": 0,
  "total_cast": 0
}"""

results = []
print(f"🚀 Starting Qwen2.5-VL Audit on {len(test_batch)} real result sheets...")

for idx, item in enumerate(tqdm(test_batch, desc="Auditing Polling Units")):
    try:
        # Download image directly from INEC CDN
        img_res = requests.get(item["imageUrl"], timeout=15)
        if img_res.status_code != 200:
            continue
        img = Image.open(BytesIO(img_res.content)).convert("RGB")
        
        # Prepare prompt
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": img},
                    {"type": "text", "text": SYSTEM_PROMPT}
                ]
            }
        ]
        
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        image_inputs, video_inputs = process_vision_info(messages)
        inputs = processor(
            text=[text],
            images=image_inputs,
            videos=video_inputs,
            padding=True,
            return_tensors="pt"
        ).to("cuda")
        
        with torch.no_grad():
            generated_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1)
            generated_ids_trimmed = [
                out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
            ]
            output_text = processor.batch_decode(
                generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
            )[0]
        
        # Parse JSON
        json_match = re.search(r'\{.*\}', output_text, re.DOTALL)
        if not json_match:
            continue
        parsed = json.loads(json_match.group(0))
        
        # Score against human ground truth
        gt_votes = item["groundTruth"]
        ai_votes = parsed.get("votes", {})
        
        correct = 0
        hallucinations = 0
        refusals = 0
        evaluated = 0
        discrepancies = {}
        
        for p in PARTIES:
            gt_val = gt_votes.get(p, 0)
            ai_val = ai_votes.get(p)
            evaluated += 1
            
            if ai_val is None:
                refusals += 1
            elif ai_val == gt_val:
                correct += 1
            else:
                hallucinations += 1
                discrepancies[p] = {"ai": ai_val, "gt": gt_val, "diff": ai_val - gt_val}
                
        acc_pct = (correct / evaluated) * 100 if evaluated > 0 else 0
        
        # Section 51 Statutory Check (Votes Cast <= Accredited)
        sec51_violation = False
        if parsed.get("total_cast") and parsed.get("accredited"):
            sec51_violation = parsed["total_cast"] > parsed["accredited"]
            
        # Section 64 Statutory Check (Sum of Party Votes == Valid Votes)
        calc_party_sum = sum([v for v in ai_votes.values() if isinstance(v, (int, float))])
        sec64_balanced = True
        if parsed.get("valid") is not None:
            sec64_balanced = (calc_party_sum == parsed["valid"])
            
        results.append({
            "sheetId": idx + 1,
            "puCode": item["puCode"],
            "puName": item["puName"],
            "lga": item["lga"],
            "ward": item["ward"],
            "accuracyPercent": acc_pct,
            "correctParties": correct,
            "hallucinations": hallucinations,
            "refusals": refusals,
            "sec51Violation": sec51_violation,
            "sec64Balanced": sec64_balanced,
            "discrepanciesCount": len(discrepancies),
            "aiExtracted": parsed,
            "groundTruth": gt_votes
        })
        
        # Incremental checkpoint save every 25 sheets
        if (idx + 1) % 25 == 0:
            pd.DataFrame(results).to_csv("qwen_ec8a_interim_results.csv", index=False)
            print(f"\n Checkpoint: {idx + 1} sheets completed. Current Avg Accuracy: {pd.DataFrame(results)['accuracyPercent'].mean():.2f}%")
            
    except Exception as e:
        print(f"Error on sheet {idx + 1}: {str(e)[:100]}")

# Save final results
df_results = pd.DataFrame(results)
df_results.to_csv("qwen_ec8a_final_results.csv", index=False)
with open("qwen_ec8a_final_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("\n Audit Completed Successfully!")

## Step 5: Generate Forensic Scorecard & Visualizations
*Computes macro accuracy, hallucination distribution, and Electoral Act statutory audit findings.*

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

total_sheets = len(df_results)
mean_acc = df_results['accuracyPercent'].mean()
total_eval_points = total_sheets * 15
total_hallucinations = df_results['hallucinations'].sum()
total_refusals = df_results['refusals'].sum()
total_correct = df_results['correctParties'].sum()

print("=" * 60)
print("🏆 QWEN2.5-VL FORENSIC AUDIT SCORECARD (NIGERIA FORM EC8A)")
print("=" * 60)
print(f"Total Authentic Result Sheets Audited: {total_sheets}")
print(f"Total Handwritten Data Points Tested:  {total_eval_points}")
print(f"Overall Exact Match Accuracy:           {mean_acc:.2f}%")
print(f"Total Correct Party Counts:             {total_correct} / {total_eval_points} ({(total_correct/total_eval_points)*100:.2f}%)")
print(f"Total AI Hallucinations:               {total_hallucinations} ({(total_hallucinations/total_eval_points)*100:.2f}%)")
print(f"Total Honest Refusals:                  {total_refusals} ({(total_refusals/total_eval_points)*100:.2f}%)")
print(f"Sheets with 100% Perfect Precision:    {(df_results['accuracyPercent'] == 100).sum()} / {total_sheets} ({((df_results['accuracyPercent'] == 100).sum()/total_sheets)*100:.1f}%)")
print(f"Section 51 Overvoting Flags:            {df_results['sec51Violation'].sum()} sheets")
print(f"Section 64 Mathematical Balance Rate:   {((df_results['sec64Balanced'].sum())/total_sheets)*100:.1f}%")
print("=" * 60)

# Plot Distribution
plt.figure(figsize=(10, 5))
sns.histplot(df_results['accuracyPercent'], bins=20, kde=True, color="#10B981")
plt.title("Qwen2.5-VL Exact Match Accuracy Distribution across Polling Units", fontsize=14, fontweight="bold")
plt.xlabel("Accuracy (%)")
plt.ylabel("Number of Result Sheets")
plt.grid(True, alpha=0.3)
plt.show()

## Step 6: Download Results File
*Click below to download the CSV and JSON files directly to your computer.*

In [ ]:
from google.colab import files
files.download('qwen_ec8a_final_results.csv')
files.download('qwen_ec8a_final_results.json')
print("Downloaded results to your laptop! You can share them with Antigravity to write the paper.")